# 06 - TF-IDF + Ridge

In [ ]:
import sys
sys.path.append("..")

import numpy as np

from experiments.data import load_data
from experiments.evaluation import select_candidate, record_result, diagnostics, comparison_table, inverse_log
from experiments.estimators import text_pipeline, text_inputs

In [ ]:
parts, manifest = load_data("data/processed/model_input.parquet")
train, val, test = parts["train"], parts["validation"], parts["test"]

y_train, y_val = train.price.to_numpy(), val.price.to_numpy()
log_y_train, log_y_val = np.log1p(y_train), np.log1p(y_val)

manifest["split_counts"]

## Helper

`run` fits each set of parameters, keeps the one with the lowest validation MAE, then scores it on the test set and saves the results to `reports/experiments/`.

In [ ]:
FORMAT = {"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.4f}", "MAPE": "{:.2f}"}


def run(key, candidates, fit, predict, X_val, X_test, extra=None):
    model, params, tuning = select_candidate(candidates, fit, y_val, lambda m: predict(m, X_val))
    display(tuning)
    print("Best params:", params)

    val_pred = predict(model, X_val)
    test_pred = predict(model, X_test)
    _, scores = record_result(key, manifest, parts, val_pred, test_pred, params, tuning, extra=extra)
    display(scores.style.format(FORMAT))
    diagnostics(key, test.price, test_pred)
    return model

## Text input

Joins `name` and `description`, then removes sentences that mention a price and words that contain digits.

In [ ]:
text_train = text_inputs(train)
text_val = text_inputs(val)
text_test = text_inputs(test)

text_train.head()

## Model

In [ ]:
def fit_tfidf(params):
    return text_pipeline(**params).fit(text_train, log_y_train)


def predict_tfidf(model, X):
    return inverse_log(model.predict(X), y_train)


tfidf_params = [
    {"ngram_range": (1, 1), "max_features": 30000, "min_df": 3, "max_df": 0.95, "alpha": 10.0},
    {"ngram_range": (1, 1), "max_features": 50000, "min_df": 5, "max_df": 1.0, "alpha": 100.0},
    {"ngram_range": (1, 2), "max_features": 30000, "min_df": 3, "max_df": 0.95, "alpha": 10.0},
    {"ngram_range": (1, 2), "max_features": 50000, "min_df": 5, "max_df": 1.0, "alpha": 100.0},
]
tfidf = run("tfidf", tfidf_params, fit_tfidf, predict_tfidf, text_val, text_test)

## Compare models

In [ ]:
comparison_table(manifest).style.format(FORMAT)